[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-01-depth-stability-activations.ipynb)

# Advanced Discussion 1 — why depth needs non-linearity, numerical stability, dead units, parameter counts

Reproduces the chapter 1 advanced discussion: linear collapse, XOR with two ReLU units, width versus approximation error, unstable versus stable losses, dead ReLUs, and parameter accounting.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: everything is generated locally.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
np.random.seed(0); torch.manual_seed(0)

## 1. why non-linearity: stacked linear layers collapse into one

In [ ]:
W1, b1 = np.random.randn(5, 8), np.random.randn(8)
W2, b2 = np.random.randn(8, 3), np.random.randn(3)
x = np.random.randn(4, 5)
two_layers = (x @ W1 + b1) @ W2 + b2
Wc, bc = W1 @ W2, b1 @ W2 + b2                       # the single equivalent layer
print("two linear layers == one linear layer:", np.allclose(two_layers, x @ Wc + bc))

## 2. XOR needs a hidden layer: hand-set weights, two ReLU units

In [ ]:
relu = lambda z: np.maximum(0, z)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
W_h = np.array([[1.0, 1.0], [1.0, 1.0]]); b_h = np.array([0.0, -1.0])       # h1 = x1 + x2, h2 = max(0, x1 + x2 - 1)
w_o = np.array([1.0, -2.0])                                                 # y = h1 - 2 * h2
print("XOR by a 2-unit ReLU layer:", (relu(X @ W_h + b_h) @ w_o).tolist())

## 3. width buys approximation power: fit sin(x) with n random ReLU features + least squares

In [ ]:
xs = np.linspace(-np.pi, np.pi, 400)[:, None]; ys = np.sin(2 * xs[:, 0])
for n in (1, 2, 4, 8, 16, 64):
    rng = np.random.RandomState(0)
    w, b = rng.randn(1, n) * 2, rng.uniform(-np.pi, np.pi, n) * 1.0
    H = np.maximum(0, xs @ w + b); H = np.c_[H, np.ones(len(xs))]
    coef, *_ = np.linalg.lstsq(H, ys, rcond=None)
    print("width %2d  RMSE %.4f" % (n, np.sqrt(np.mean((H @ coef - ys) ** 2))))

## 4. numerical stability: BCE from probabilities vs from logits

In [ ]:
z = torch.tensor([-120.0, -30.0, 0.0, 30.0, 120.0], dtype=torch.float32); y = torch.tensor([1.0, 1.0, 1.0, 0.0, 0.0])
p = torch.sigmoid(z)
naive = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
stable = F.binary_cross_entropy_with_logits(z, y, reduction="none")
print("logit  ", z.tolist()); print("naive  ", naive.tolist()); print("stable ", [round(v, 3) for v in stable.tolist()])
def softmax_naive(a): e = np.exp(a); return e / e.sum()
def softmax_stable(a): e = np.exp(a - a.max()); return e / e.sum()
a = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    print("softmax naive :", softmax_naive(a)); print("softmax stable:", np.round(softmax_stable(a), 4))

## 5. activations: dead ReLUs after one destructive update

In [ ]:
torch.manual_seed(0)
Xb = torch.randn(1000, 20)
lin = nn.Linear(20, 100)
with torch.no_grad(): frac0 = (F.relu(lin(Xb)) == 0).all(0).float().mean().item()
with torch.no_grad(): lin.bias.sub_(4.0)                                  # a big, unlucky update pushes pre-activations negative
with torch.no_grad(): dead_relu = (F.relu(lin(Xb)) == 0).all(0).float().mean().item(); dead_leaky = (F.leaky_relu(lin(Xb), 0.01) == 0).all(0).float().mean().item()
print("units that never fire: at init %.2f | after bias shift ReLU %.2f | LeakyReLU %.2f" % (frac0, dead_relu, dead_leaky))
g = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print("GELU", [round(v, 3) for v in F.gelu(g).tolist()], "SiLU", [round(v, 3) for v in F.silu(g).tolist()])

## 6. parameter accounting for the Meridian MLP (23 features -> 64 -> 32 -> 1)

In [ ]:
m = nn.Sequential(nn.Linear(23, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
for name, p in m.named_parameters(): print("%-10s %-12s %6d" % (name, tuple(p.shape), p.numel()))
print("total parameters:", sum(p.numel() for p in m.parameters()))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.